# 情感分析模型训练

TF-IDF + 逻辑回归，13 类合并为 3 类：`positive` / `negative` / `neutral`

**环境**：Anaconda 

In [1]:
import json
import os
from datetime import datetime

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from config import (
    BASE_DIR,
    DATASET_PATH,
    MODEL_ARTIFACTS_PREFIX,
    RANDOM_STATE,
    TEST_SIZE,
    TFIDF_MODEL_PATH,
)
from utils import clean_text, load_stopwords, merge_label

In [2]:
stopwords = load_stopwords()
df = pd.read_csv(DATASET_PATH)
df["text"] = df["content"].apply(lambda t: clean_text(t, stopwords))
df["sentiment"] = df["sentiment"].apply(merge_label)

print("标签分布:")
print(df["sentiment"].value_counts())
df.head()

标签分布:
sentiment
negative    16063
positive    15299
neutral      8638
Name: count, dtype: int64


,tweet_id,sentiment,content,text
0,1956967341,negative,@tiffanylue i know i was listenin to bad habi...,listenin bad habit earlier started freakin
1,1956967666,negative,Layin n bed with a headache ughhhh...waitin o...,layin bed headache ughhhhwaitin call
2,1956967696,negative,Funeral ceremony...gloomy friday...,funeral ceremonygloomy friday
3,1956967789,positive,wants to hang out with friends SOON!,hang friends
4,1956968416,neutral,@dannycastillo We want to trade with someone w...,trade houston tickets


In [3]:
x_train, x_test, y_train, y_test = train_test_split(
    df["text"],
    df["sentiment"],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["sentiment"],
)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20_000,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        C=1.0,
        class_weight="balanced",
    )),
])

pipeline.fit(x_train, y_train)
y_pred = pipeline.predict(x_test)

print(classification_report(y_test, y_pred))
report = classification_report(y_test, y_pred, output_dict=True)
print(f"Accuracy: {report['accuracy']:.4f}")
print(f"Macro F1: {report['macro avg']['f1-score']:.4f}")

              precision    recall  f1-score   support

    negative       0.63      0.56      0.59      3213
     neutral       0.33      0.47      0.39      1727
    positive       0.61      0.54      0.57      3060

    accuracy                           0.53      8000
   macro avg       0.52      0.52      0.52      8000
weighted avg       0.56      0.53      0.54      8000

Accuracy: 0.5325
Macro F1: 0.5179


In [4]:
joblib.dump(pipeline, TFIDF_MODEL_PATH)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = os.path.join(BASE_DIR, f"{MODEL_ARTIFACTS_PREFIX}{timestamp}")
os.makedirs(save_dir, exist_ok=True)
joblib.dump(pipeline, os.path.join(save_dir, "tfidf_model.pkl"))

with open(os.path.join(save_dir, "classification_report.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4)

print(f"模型已保存: {TFIDF_MODEL_PATH}")
print(f"备份目录: {save_dir}")

模型已保存: d:\BaiduNetdiskDownload\情感分析\情感分析\tfidf_model.pkl
备份目录: d:\BaiduNetdiskDownload\情感分析\情感分析\model_artifacts_20260710_042823
